# Publish the fine-tuned hematology adapter on Hugging Face

Uploads the **new** LoRA adapter (trained on the 90 % split and evaluated in the thesis) from Google Drive to your Hugging Face account, writes a model card, and makes the repository **public**.

* Runs on a **CPU** runtime — no GPU needed. Takes ~5 minutes.
* You need a Hugging Face **write** token: huggingface.co → Settings → Access Tokens → *Create new token* → type **Write**.
* The token is typed into a hidden box and is **never printed**. Do not paste it into a code cell.

In [ ]:
HF_USERNAME  = 'Afridi07'                          # your Hugging Face username
REPO_NAME    = 'hematology-llama-3.1-8b-lora'      # same repo as before: the new adapter replaces the old files
DRIVE_DIR    = '/content/drive/MyDrive/thesis_domain_expert'
ADAPTER_DIR  = f'{DRIVE_DIR}/hema-llama-lora'
MAKE_PUBLIC  = True

In [ ]:
%pip install -q -U huggingface_hub
import os, json, csv, getpass
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(ADAPTER_DIR), f'adapter folder not found: {ADAPTER_DIR}'
print('adapter files:', sorted(os.listdir(ADAPTER_DIR)))

In [ ]:
from huggingface_hub import HfApi, login
token = getpass.getpass('Paste your Hugging Face WRITE token (hidden): ').strip()
login(token=token, add_to_git_credential=False)
api = HfApi(token=token)
print('Logged in as:', api.whoami()['name'])
repo_id = f'{HF_USERNAME}/{REPO_NAME}'
api.create_repo(repo_id, repo_type='model', exist_ok=True, private=not MAKE_PUBLIC)
print('Repository ready:', repo_id)

In [ ]:
# Read the evaluation results produced by domain_expert_evaluation.ipynb
rows = list(csv.reader(open(f'{DRIVE_DIR}/results_summary.csv')))
table = '| System | ' + ' | '.join(rows[0][1:]) + ' |\n|' + '---|' * len(rows[0]) + '\n'
table += '\n'.join('| ' + ' | '.join(r) + ' |' for r in rows[1:])
cfg = json.load(open(f'{DRIVE_DIR}/evaluation_report.json'))['config']

card = f'''---
base_model: meta-llama/Llama-3.1-8B-Instruct
library_name: peft
license: llama3.1
language: [en]
tags: [lora, qlora, unsloth, hematology, medical, bsc-thesis]
---

# Hematology domain-expert adapter for Llama-3.1-8B-Instruct (QLoRA)

LoRA adapter produced for the BSc thesis *A Hybrid Multimodal Lab Assistant: Blood Image Classification
Combined with Domain-Expert LLM Reasoning and RAG* (University of Debrecen, Faculty of Informatics, 2026).
It adapts Llama-3.1-8B-Instruct to answer hematology questions concisely, in the style of textbook-grounded answers.

**Research artefact only. Not a medical device and not for clinical use.**

## Training
* Base: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (4-bit), QLoRA, rank 16, alpha 32, all attention and MLP projections
  (41.9 M trainable parameters, 0.52 %).
* Data: {cfg["n_train"]} synthetic question-answer pairs generated by an OpenAI GPT model from passages of two hematology
  textbooks (the data are not redistributed because the source books are copyrighted).
* {cfg["epochs"]} epochs, batch 8, learning rate 2e-4, seed {cfg["seed"]}, one NVIDIA T4 (about 50 minutes).

## Evaluation (from the thesis)
Held-out textbook questions (chunk-level split, {cfg["n_indomain_scored"]} scored) and {cfg["n_mcq"]} hematology-related
MedMCQA questions; base = the same model with the adapter disabled.

{table}

**Interpretation.** The adapter makes answers much closer to textbook reference answers but does **not** improve
exam-question accuracy: it teaches answering style, not new knowledge. It appends a line such as
"Source: essentials_haematology.pdf." to every answer, **including incorrect ones** -- this line is learned
format, not a verified citation. Use it together with retrieval (RAG) over real sources.

## Usage
```python
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("{repo_id}", load_in_4bit=True)
```
or serve it with vLLM (`--enable-lora`) behind an OpenAI-compatible endpoint.

## License
Built with Llama. Use is subject to the Llama 3.1 Community License.
'''
open(f'{ADAPTER_DIR}/README.md', 'w').write(card)
print(card[:1500])

In [ ]:
api.upload_folder(repo_id=repo_id, folder_path=ADAPTER_DIR, commit_message='Adapter trained on 90% chunk-level split (thesis version)')
for f in ['training_loss.png', 'results_summary.csv', 'results_chart.png']:
    p = f'{DRIVE_DIR}/{f}'
    if os.path.exists(p): api.upload_file(path_or_fileobj=p, path_in_repo=f'eval/{f}', repo_id=repo_id)
if MAKE_PUBLIC:
    try:
        api.update_repo_settings(repo_id=repo_id, private=False)
    except AttributeError:
        api.update_repo_visibility(repo_id=repo_id, private=False)
info = api.model_info(repo_id)
print('private:', info.private)
print('files:', [s.rfilename for s in info.siblings])
print(f'\nDone: https://huggingface.co/{repo_id}')

**Check:** open the link printed above in a private/incognito browser window. If the page loads without logging in, the model is public.
The previous adapter (trained on all 2,098 pairs) is still in the repository's commit history, but the current files are the new version.